# 05 — Validación contable
Se comprueban las identidades que el propio anuario debe cumplir. Cada incumplimiento va a
`discrepancias.csv` con su cuadro y página. **Las discrepancias no se corrigen**: son hallazgos
sobre la fuente. Además se corren controles técnicos del ETL (geografía, juzgados, contexto y
correcciones de tipo de proceso) que sí detienen el pipeline si fallan.

| # | identidad |
|---|---|
| 1 | atendidas = pendientes_inicio + ingresadas |
| 2 | atendidas = resueltas + pendientes_fin |
| 3 | fila de total = suma de las filas de dato |
| 4 | 4.1.x: última columna = suma de las demás |
| 5 | 14.1.x: mujer + varón + acefalías = total |
| 6 | 14.1.x: cada SUB TOTAL del 14.1.1 = su fila del 14.1.3 |
| 7 | 13.1.x: atendidas = pendientes + recibidas; suma de faltas = total de sanciones |
| 8 | capitales (9.1.1) + provincias (9.1.5) = consolidado (9.1.9) |
| 9 | cap. 5 y 6: suma de formas de ingreso = atendidas |
| 10 | cap. 5 y 6: atendidas = formas de salida + pendientes |
| 11 | cap. 5 y 6: TOTAL de cada ciudad = suma de sus tipos de proceso |
| 12 | total nacional de cada cuadro de los cap. 5 y 6 = su materia en el 9.1.1 / 9.1.5 |

In [ ]:
%run -i modulos/comun.ipynb
%run -i modulos/geografia.ipynb
%run -i modulos/juzgados.ipynb
%run -i modulos/procesos.ipynb
%run -i modulos/contexto_procesos.ipynb
%run -i modulos/correcciones_tipo_proceso.ipynb
import matplotlib.pyplot as plt

TOLERANCIA = 1e-9
discrepancias = []
TABLAS_PROCESOS = {
    "causas_por_tipo_proceso": "norm_procesos_causas.csv",
    "resueltas_por_tipo_proceso": "norm_procesos_resueltas.csv",
    "apelaciones_por_tipo_proceso": "norm_procesos_apelacion.csv",
    "ejecucion_por_tipo_proceso": "norm_procesos_ejecucion.csv",
    "otros_tramites_por_tipo_proceso": "norm_procesos_otros.csv",
}
# El número de juzgados es un atributo de la ciudad y la rebeldía una anotación: no entran al balance.
FUERA_DEL_BALANCE = ["num_juzgados", "procesos_rebeldia"]


def registrar(identidad, cuadro, pagina, entidad, calculado, publicado, detalle=""):
    if calculado is None or publicado is None:
        return
    diferencia = calculado - publicado
    if abs(diferencia) <= TOLERANCIA:
        return
    discrepancias.append({"identidad": identidad, "cuadro_origen": cuadro, "pagina_pdf": pagina, "entidad": entidad,
                          "calculado": calculado, "publicado": publicado, "diferencia": diferencia, "detalle": detalle})


def registrar_no_comparable(identidad, cuadro, pagina, entidad, detalle):
    discrepancias.append({"identidad": identidad, "cuadro_origen": cuadro, "pagina_pdf": pagina, "entidad": entidad,
                          "calculado": None, "publicado": None, "diferencia": None, "detalle": detalle})


def v(fila, columna):
    valor = fila.get(columna)
    if valor is None or pd.isna(valor):
        return None
    return float(valor)


def control(lista, nombre, esperado, observado):
    # Agrega un control a la lista y devuelve 1 si falló.
    if observado == esperado:
        estado = "OK"
    else:
        estado = "FALLO"
    lista.append({"control": nombre, "esperado": esperado, "observado": observado, "estado": estado})
    if estado == "FALLO":
        return 1
    return 0


def totales_por_cuadro(df, columnas, nombre):
    # Identidad 3
    for cuadro, g in df.groupby("cuadro_origen"):
        datos = g[g["tipo_fila_derivado"] == "dato"]
        totales = g[g["tipo_fila_derivado"] == "total"]
        if datos.empty or totales.empty:
            continue
        fila_total = totales.iloc[0]
        for col in columnas:
            if col not in g.columns:
                continue
            suma = datos[col].sum(skipna=True)
            registrar("total = suma de filas (" + col + ")", cuadro, fila_total["pagina_pdf"],
                      fila_total.get("etiqueta_fila", "TOTAL"), float(suma), v(fila_total, col), nombre)


def identidades_de_movimiento(df, nombre):
    # Identidades 1 y 2
    for i, r in df.iterrows():
        pi = v(r, "pendientes_inicio")
        ing = v(r, "ingresadas")
        at = v(r, "atendidas")
        res = v(r, "resueltas")
        pf = v(r, "pendientes_fin")
        etiqueta = r.get("etiqueta_fila")
        if not etiqueta:
            etiqueta = r.get("gestion")
        if pi is not None and ing is not None and at is not None:
            registrar("atendidas = pendientes_inicio + ingresadas", r["cuadro_origen"], r["pagina_pdf"], etiqueta, pi + ing, at, nombre)
        if res is not None and pf is not None and at is not None:
            registrar("atendidas = resueltas + pendientes_fin", r["cuadro_origen"], r["pagina_pdf"], etiqueta, res + pf, at, nombre)

## Capítulo 9: identidades 1, 2, 3 y 8

In [ ]:
mov = pd.read_csv(INTERIM / "norm_causas_movimiento.csv")
identidades_de_movimiento(mov, "9.1.x movimiento")
totales_por_cuadro(mov, ["num_juzgados", "pendientes_inicio", "ingresadas", "atendidas", "resueltas", "pendientes_fin"], "9.1.x movimiento")
his = pd.read_csv(INTERIM / "norm_causas_historico.csv")
identidades_de_movimiento(his, "9.1.x histórico")

# Identidad 8, materia por materia (por materia_norm, con la errata ya corregida).
por_cuadro = {}
for c, g in mov[mov["eje"] == "materia"].groupby("cuadro_origen"):
    por_cuadro[c] = g.set_index("materia_norm")
cap = por_cuadro["9.1.1"]
pro = por_cuadro["9.1.5"]
con = por_cuadro["9.1.9"]
for materia in con.index:
    if pd.isna(materia):
        continue
    if materia not in cap.index or materia not in pro.index:
        registrar_no_comparable("9.1.1 + 9.1.5 = 9.1.9", "9.1.9", 701, materia, "la materia no está en los tres cuadros; no se compara")
        continue
    for col in ["pendientes_inicio", "ingresadas", "atendidas", "resueltas", "pendientes_fin"]:
        a = v(cap.loc[materia], col)
        b = v(pro.loc[materia], col)
        c = v(con.loc[materia], col)
        if a is None or b is None or c is None:
            continue
        registrar("9.1.1 + 9.1.5 = 9.1.9 (" + col + ")", "9.1.9", 701, materia, a + b, c, "cruce entre ámbitos")
print(len(discrepancias), "discrepancias hasta acá")

## 4.1 — Juzgados: identidad 4 y controles de la semántica auditada
En el 4.1.1 la suma ingenua de la columna TOTAL cuenta dos veces cada juzgado (subtotales +
detalle); solo las hojas de la jerarquía dan el total real.

In [ ]:
largo = pd.read_csv(INTERIM / "norm_juzgados.csv")
clave = ["cuadro_origen", "pagina_pdf", "fila_en_cuadro", "provincia_o_grupo", "localidad_o_subtipo"]
for llave, g in largo.groupby(clave, dropna=False):
    cuadro = llave[0]
    pagina = llave[1]
    total = g[g["es_ultima_columna"]]
    resto = g[~g["es_ultima_columna"]]
    if total.empty:
        continue
    # Los cuadros provinciales traen población en col_01: no entra en la suma.
    if cuadro != "4.1.1":
        resto = resto[resto["columna"] != "col_01"]
    partes = []
    for x in [llave[3], llave[4]]:
        if isinstance(x, str):
            partes.append(str(x))
    registrar("última columna = suma de la fila", cuadro, pagina, " / ".join(partes),
              float(resto["valor"].sum(skipna=True)), float(total["valor"].iloc[0]), "4.1.x")

validaciones_juzgados = []
errores_juzgados = 0
indeterminados = 0
for k in ENCABEZADOS_JUZGADOS:
    if ENCABEZADOS_JUZGADOS[k]["tipo_columna"] == "indeterminado":
        indeterminados = indeterminados + 1
errores_juzgados += control(validaciones_juzgados, "claves de encabezados auditadas", 130, len(ENCABEZADOS_JUZGADOS))
errores_juzgados += control(validaciones_juzgados, "filas auditadas de 4.1.1", 37, len(FILAS_4_1_1))
errores_juzgados += control(validaciones_juzgados, "filas de juzgados", 1148, len(largo))
errores_juzgados += control(validaciones_juzgados, "filas fuente de 4.1.1", 37, largo.loc[largo["cuadro_origen"] == "4.1.1", "fila_en_cuadro"].nunique())
errores_juzgados += control(validaciones_juzgados, "casos indeterminados de columna", 1, indeterminados)
errores_juzgados += control(validaciones_juzgados, "celdas de Tarija 4.1.7/col_09", 3, len(largo[(largo["cuadro_origen"] == "4.1.7") & (largo["columna"] == "col_09")]))
errores_juzgados += control(validaciones_juzgados, "celdas sin tipo_columna", 0, int(largo["tipo_columna"].isna().sum()))

cap411 = largo[largo["cuadro_origen"] == "4.1.1"]


def celda(fila_id, columna):
    valores = cap411[(cap411["fila_id"] == fila_id) & (cap411["columna"] == columna)]["valor"].dropna()
    if len(valores) > 1:
        return None
    if valores.empty:
        return 0.0
    return float(valores.iloc[0])


def hijos_de(padre):
    hijos = []
    for d in FILAS_4_1_1.values():
        if d["fila_padre_id"] == padre:
            hijos.append(d["fila_id"])
    return hijos


columnas_411 = []
for i in range(1, 12):
    columnas_411.append("col_" + str(i).zfill(2))
coincidencias_subtotales = 0
for d in FILAS_4_1_1.values():
    if d["estructura"] != "subtotal":
        continue
    for columna in columnas_411:
        calculado = 0
        for hijo in hijos_de(d["fila_id"]):
            calculado = calculado + celda(hijo, columna)
        if celda(d["fila_id"], columna) == calculado:
            coincidencias_subtotales = coincidencias_subtotales + 1
errores_juzgados += control(validaciones_juzgados, "subtotales de 4.1.1 coincidentes", 55, coincidencias_subtotales)

coincidencias_total = 0
for columna in columnas_411:
    calculado = 0
    for hijo in hijos_de("f037"):
        calculado = calculado + celda(hijo, columna)
    if celda("f037", columna) == calculado:
        coincidencias_total = coincidencias_total + 1
errores_juzgados += control(validaciones_juzgados, "total general de 4.1.1 coincidente", 11, coincidencias_total)

total_publicado = cap411[cap411["columna"] == "col_11"]
suma_ingenua = int(total_publicado["valor"].sum())
suma_hojas = int(total_publicado[total_publicado["es_hoja_jerarquia"].fillna(False)]["valor"].sum())
errores_juzgados += control(validaciones_juzgados, "suma ingenua de TOTAL 4.1.1", 2422, suma_ingenua)
errores_juzgados += control(validaciones_juzgados, "suma de hojas de TOTAL 4.1.1", 846, suma_hojas)
errores_juzgados += control(validaciones_juzgados, "exceso por doble conteo de TOTAL 4.1.1", 1576, suma_ingenua - suma_hojas)
pd.DataFrame(validaciones_juzgados).to_csv(INTERIM / "validacion_juzgados.csv", index=False, encoding="utf-8")
pd.DataFrame(validaciones_juzgados)

## 14.1 — Personal (identidades 5 y 6) y 13.1 — Autoridad sumariante (identidad 7)

In [ ]:
per = pd.read_csv(INTERIM / "norm_personal.csv")
for i, r in per.iterrows():
    partes_entidad = []
    for x in [r.get("distrito"), r.get("ente")]:
        if isinstance(x, str):
            partes_entidad.append(x)
    entidad = " / ".join(partes_entidad)
    for prefijo in ["items", "remun"]:
        partes = [v(r, prefijo + "_mujer"), v(r, prefijo + "_varon"), v(r, prefijo + "_acefalias")]
        total = v(r, prefijo + "_total")
        presentes = []
        for p in partes:
            if p is not None:
                presentes.append(p)
        if total is None or len(presentes) == 0:
            continue
        registrar(prefijo + ": mujer + varón + acefalías = total", r["cuadro_origen"], r["pagina_pdf"], entidad, sum(presentes), total, "14.1.x")

sub = per[(per["cuadro_origen"] == "14.1.1") & (per["ente"] == "SUB TOTAL")]
tot = per[per["cuadro_origen"] == "14.1.3"].set_index("distrito")
# El 14.1.1 llama NACIONAL a lo que el 14.1.3 llama OFICINA NACIONAL.
alias = {"NACIONAL": "OFICINA NACIONAL"}
for i, r in sub.iterrows():
    clave_distrito = alias.get(r["distrito"], r["distrito"])
    if clave_distrito not in tot.index:
        continue
    for col in ["items_total", "remun_total"]:
        registrar("SUB TOTAL de 14.1.1 = fila de 14.1.3 (" + col + ")", "14.1.1", r["pagina_pdf"], r["distrito"],
                  v(r, col), v(tot.loc[clave_distrito], col), "cruce 14.1.1 / 14.1.3")

In [ ]:
sum_ = pd.read_csv(INTERIM / "norm_sumariante.csv")
for i, r in sum_.iterrows():
    if r["cuadro_origen"] == "13.1.1":
        pi = v(r, "pendientes_inicio")
        rec = v(r, "recibidas")
        at = v(r, "atendidas")
        if pi is not None and rec is not None and at is not None:
            registrar("atendidas = pendientes + recibidas", r["cuadro_origen"], r["pagina_pdf"], r["etiqueta_fila"], pi + rec, at, "13.1.1")
    else:
        presentes = []
        for c in ["amonestacion", "multa", "suspension", "destitucion"]:
            if v(r, c) is not None:
                presentes.append(v(r, c))
        total = v(r, "total_sanciones")
        if total is not None and len(presentes) > 0:
            registrar("suma de faltas = total de sanciones", r["cuadro_origen"], r["pagina_pdf"], r["etiqueta_fila"], sum(presentes), total, "13.1.2 / 13.1.3")
totales_por_cuadro(sum_, ["pendientes_inicio", "recibidas", "atendidas", "amonestacion", "multa", "suspension",
                          "destitucion", "total_sanciones"], "13.1.x")
print(len(discrepancias), "discrepancias hasta acá")

## Capítulos 5 y 6: identidades 9 a 12
Las columnas anteriores a "atendidas" en el layout son formas de ingreso; las posteriores, de salida.

In [ ]:
causas = pd.read_csv(INTERIM / "norm_procesos_causas.csv", low_memory=False)
for firma, g in causas.groupby("firma"):
    nombres = LAYOUTS_PROCESOS[firma]["columnas"]
    if "atendidas" not in nombres:
        continue
    corte = nombres.index("atendidas")
    entran = []
    for n in nombres[:corte]:
        if n not in FUERA_DEL_BALANCE:
            entran.append(n)
    salen = []
    for n in nombres[corte + 1:]:
        if n not in FUERA_DEL_BALANCE:
            salen.append(n)
    for i, r in g[g["unidad_fila"] == "tipo_proceso"].iterrows():
        etiqueta = str(r["entidad"]) + " / " + str(r["tipo_proceso"])
        registrar("suma de formas de ingreso = atendidas", r["cuadro_origen"], r["pagina_pdf"], etiqueta,
                  float(r[entran].sum()), v(r, "atendidas"), "capítulos 5 y 6")
        registrar("atendidas = formas de salida + pendientes", r["cuadro_origen"], r["pagina_pdf"], etiqueta,
                  float(r[salen].sum()), v(r, "atendidas"), "capítulos 5 y 6")

NO_VALORES = ["cuadro_origen", "pagina_pdf", "firma", "familia", "ambito", "titulo_pagina", "entidad",
              "num_juzgados_pagina", "unidad_fila", "grupo_proceso", "grupo_proceso_norm", "tipo_proceso_extraido",
              "tipo_proceso", "tipo_fila_derivado", "orden_fila", "n_columnas", "materia_seccion", "materia_norm",
              "materia_cruda", "ciudad", "distrito", "departamento_derivado", "es_total_nacional", "gestion",
              "revisado_manual", "columna", "orden_columna", "rotulo_columna_pdf", "tipo_accion_penal",
              "etapa_proceso_fuente", "contexto_accion_penal"]


def totales_por_entidad(df, familia):
    # Identidad 11
    valores = []
    for c in df.columns:
        if c not in NO_VALORES and c not in FUERA_DEL_BALANCE and pd.api.types.is_numeric_dtype(df[c]):
            valores.append(c)
    for llave, g in df.groupby(["cuadro_origen", "pagina_pdf", "entidad"], sort=False):
        cuadro, pagina, entidad = llave
        total = g[g["tipo_fila_derivado"] == "total"]
        detalle = g[g["tipo_fila_derivado"] == "detalle"]
        if len(total) != 1 or detalle.empty:
            continue
        fila = total.iloc[0]
        for col in valores:
            if pd.isna(fila.get(col)) and detalle[col].isna().all():
                continue
            registrar("TOTAL " + str(entidad) + " = suma de tipos de proceso (" + col + ")", cuadro, pagina, entidad,
                      float(detalle[col].sum(skipna=True)), v(fila, col), "capítulos 5 y 6, familia " + familia)


totales_por_entidad(causas, "causas")
for familia in ["resueltas", "apelacion", "ejecucion", "otros"]:
    largo_familia = pd.read_csv(INTERIM / ("norm_procesos_" + familia + ".csv"), low_memory=False)
    ancho = largo_familia.pivot_table(index=["cuadro_origen", "pagina_pdf", "entidad", "tipo_fila_derivado", "orden_fila"],
                                      columns="columna", values="valor", aggfunc="sum").reset_index()
    totales_por_entidad(ancho, familia)

Identidad 12, el cruce entre capítulos: las mismas cifras publicadas dos veces, en capítulos
distintos y con desgloses distintos. Es la prueba de que el parser lee bien. Se suman las filas de
detalle porque en penal un cuadro cubre tres materias del 9.1.x.

In [ ]:
for cuadro in CRUCE_CON_CUADRO_9:
    g = causas[(causas["cuadro_origen"] == cuadro) & causas["es_total_nacional"] & (causas["tipo_fila_derivado"] == "detalle")]
    if g.empty:
        continue
    if cuadro.startswith("5."):
        referencia = "9.1.1"
    else:
        referencia = "9.1.5"
    for materia, filas in g.groupby("materia_norm"):
        nombres = LAYOUTS_PROCESOS[filas.iloc[0]["firma"]]["columnas"]
        if "atendidas" not in nombres:
            continue
        corte = nombres.index("atendidas")
        entran = []
        for n in nombres[:corte]:
            if n not in FUERA_DEL_BALANCE:
                entran.append(n)
        fila9 = mov[(mov["cuadro_origen"] == referencia) & (mov["materia_norm"] == materia)]
        if fila9.empty:
            registrar_no_comparable("cruce con el capítulo 9", cuadro, filas.iloc[0]["pagina_pdf"], materia,
                                    "la materia no aparece en el cuadro " + referencia)
            continue
        fila9 = fila9.iloc[0]
        comparaciones = {
            "pendientes_inicio": float(filas[entran[0]].sum()),
            "ingresadas": float(filas[entran[1:]].sum().sum()),
            "atendidas": float(filas["atendidas"].sum()),
            "resueltas": None,
            "pendientes_fin": float(filas["pendientes_fin"].sum()),
        }
        if "resueltas" in nombres:
            comparaciones["resueltas"] = float(filas["resueltas"].sum())
        for col in comparaciones:
            calculado = comparaciones[col]
            if calculado is None or col not in fila9:
                continue
            registrar("total nacional de " + cuadro + " = " + referencia + " (" + col + ")", cuadro,
                      filas.iloc[0]["pagina_pdf"], materia, calculado, v(fila9, col), "cruce entre capítulos")
print(len(discrepancias), "discrepancias hasta acá")

## Controles del contexto auditado (etapa y acción penal)

In [ ]:
tablas = {}
for tabla in TABLAS_PROCESOS:
    tablas[tabla] = pd.read_csv(INTERIM / TABLAS_PROCESOS[tabla], low_memory=False)

cuadros = set()
for df in tablas.values():
    cuadros = cuadros | set(df["cuadro_origen"].astype(str).unique())


def estrato_semantico(causas):
    # Filas de detalle territoriales: su clave semántica tiene que ser única.
    estrato = causas[(causas["tipo_fila_derivado"] == "detalle") & ~causas["es_total_nacional"] & causas["tipo_proceso"].notna()].copy()
    territorio = []
    for ambito, ciudad, distrito in zip(estrato["ambito"], estrato["ciudad"], estrato["distrito"]):
        if ambito == "capital":
            territorio.append(normalizar_geografia(ciudad))
        else:
            territorio.append(normalizar_geografia(distrito))
    estrato["territorio"] = territorio
    clave = ["ambito", "territorio", "materia_homologada", "tipo_proceso", "etapa_proceso_fuente", "contexto_accion_penal"]
    return estrato, estrato.groupby(clave, dropna=False).size()


def grupos_repetidos(causas):
    # claves_repetidas_tipo_proceso.csv: filas con igual tipo de proceso que se distinguen por etapa o contexto.
    repetidas = pd.read_csv(PROCESSED / "auditoria" / "claves_repetidas_tipo_proceso.csv")
    tecnica = causas[["cuadro_origen", "pagina_pdf", "orden_fila", "etapa_proceso_fuente", "contexto_accion_penal"]]
    comprobadas = repetidas.merge(tecnica, on=["cuadro_origen", "pagina_pdf", "orden_fila"], how="left", validate="one_to_one")
    etapa = comprobadas[comprobadas["causa_repeticion"] == "subbloque_fuente"]
    contexto = comprobadas[comprobadas["causa_repeticion"] == "tipo_accion_penal"]
    etapa_ok = int((etapa.groupby("id_grupo_repetido")["etapa_proceso_fuente"].nunique() == 2).sum())
    grupos_contexto = contexto.groupby("id_grupo_repetido")
    contexto_ok = int((grupos_contexto["contexto_accion_penal"].nunique() == grupos_contexto.size()).sum())
    return etapa, contexto, etapa_ok, contexto_ok


validaciones_contexto = []
errores_contexto = 0
errores_contexto += control(validaciones_contexto, "cuadros inventariados", 95, len(cuadros))
errores_contexto += control(validaciones_contexto, "cuadros inesperados", 0, len(cuadros - CUADROS_CONOCIDOS))
errores_contexto += control(validaciones_contexto, "cuadros con etapa auditada", 6, len(ETAPA_AUDITADA_POR_CUADRO))
errores_contexto += control(validaciones_contexto, "cuadros explícitos no_aplica", 89, len(CUADROS_NO_APLICA))
nulos_etapa = 0
nulos_contexto = 0
fuera_etapa = 0
fuera_contexto = 0
dominio_etapa = set(ETAPA_POR_CUADRO.values())
dominio_contexto = set(CONTEXTOS_ACCION_PENAL) | {"no_aplica"}
for df in tablas.values():
    nulos_etapa = nulos_etapa + df["etapa_proceso_fuente"].isna().sum()
    nulos_contexto = nulos_contexto + df["contexto_accion_penal"].isna().sum()
    fuera_etapa = fuera_etapa + (~df["etapa_proceso_fuente"].isin(dominio_etapa)).sum()
    fuera_contexto = fuera_contexto + (~df["contexto_accion_penal"].isin(dominio_contexto)).sum()
errores_contexto += control(validaciones_contexto, "nulos de etapa", 0, int(nulos_etapa))
errores_contexto += control(validaciones_contexto, "nulos de contexto penal", 0, int(nulos_contexto))
errores_contexto += control(validaciones_contexto, "etapas fuera de dominio", 0, int(fuera_etapa))
errores_contexto += control(validaciones_contexto, "contextos penales fuera de dominio", 0, int(fuera_contexto))

# En formato largo el contexto pertenece a la fila fuente: cada fila fuente tiene un solo valor.
inconsistentes = 0
for tabla in ["resueltas_por_tipo_proceso", "apelaciones_por_tipo_proceso", "ejecucion_por_tipo_proceso", "otros_tramites_por_tipo_proceso"]:
    fuente = tablas[tabla].groupby(["cuadro_origen", "pagina_pdf", "orden_fila"], dropna=False)
    malas = (fuente["etapa_proceso_fuente"].nunique(dropna=False) > 1) | (fuente["contexto_accion_penal"].nunique(dropna=False) > 1)
    inconsistentes = inconsistentes + int(malas.sum())
errores_contexto += control(validaciones_contexto, "filas fuente longitudinales inconsistentes", 0, inconsistentes)

causas = tablas["causas_por_tipo_proceso"]
etapa, contexto, etapa_ok, contexto_ok = grupos_repetidos(causas)
errores_contexto += control(validaciones_contexto, "grupos por etapa diferenciados", 57, etapa_ok)
errores_contexto += control(validaciones_contexto, "filas en grupos por etapa", 114, len(etapa))
errores_contexto += control(validaciones_contexto, "grupos contexto penal diferenciados", 38, contexto_ok)
errores_contexto += control(validaciones_contexto, "filas en grupos contexto penal", 96, len(contexto))

estrato, conteos = estrato_semantico(causas)
duplicados = conteos[conteos > 1]
errores_contexto += control(validaciones_contexto, "filas estrato", 1997, len(estrato))
errores_contexto += control(validaciones_contexto, "claves semánticas únicas", 1997, len(conteos))
errores_contexto += control(validaciones_contexto, "duplicados semánticos", 0, len(duplicados))
errores_contexto += control(validaciones_contexto, "filas semánticas duplicadas", 0, int(duplicados.sum()))
errores_contexto += control(validaciones_contexto, "máxima multiplicidad", 1, int(conteos.max()))
pd.DataFrame(validaciones_contexto).to_csv(INTERIM / "validacion_contexto_procesos.csv", index=False, encoding="utf-8")
pd.DataFrame(validaciones_contexto)

## Controles de las 87 correcciones de tipo de proceso
"Fila fuente" = fila del PDF; "fila física" = fila de la tabla (en formato largo, una por valor).

In [ ]:
def distintas(izquierda, derecha):
    iguales = (izquierda == derecha) | (izquierda.isna() & derecha.isna())
    return ~iguales


def filas_fuente(df):
    grupos = df.groupby(["cuadro_origen", "pagina_pdf", "orden_fila"], sort=False, dropna=False)
    malas = (grupos["tipo_proceso_extraido"].nunique(dropna=False) > 1) | (grupos["tipo_proceso"].nunique(dropna=False) > 1)
    fuente = grupos[["tipo_proceso_extraido", "tipo_proceso"]].first().reset_index()
    return fuente, int(malas.sum())


validaciones_correcciones = []
errores_correcciones = 0
reglas = CORRECCIONES_TIPO_PROCESO
n_generales = 0
n_cuadro = 0
n_fila = 0
for r in reglas:
    if r["alcance"] == ALCANCE_CUADRO:
        n_cuadro = n_cuadro + 1
    elif r["alcance"] == ALCANCE_FILA_CONTEXTO:
        n_fila = n_fila + 1
    else:
        n_generales = n_generales + 1
errores_correcciones += control(validaciones_correcciones, "reglas auditadas", 87, len(reglas))
errores_correcciones += control(validaciones_correcciones, "reglas implementadas", 87, len(reglas))
errores_correcciones += control(validaciones_correcciones, "reglas generales", 0, n_generales)
errores_correcciones += control(validaciones_correcciones, "reglas acotadas por cuadro", 80, n_cuadro)
errores_correcciones += control(validaciones_correcciones, "reglas acotadas por fila/contexto", 7, n_fila)

fuentes = {}
reglas_aplicadas = 0
reglas_sin_match = 0
reglas_con_exceso = 0
reglas_contradictorias = 0
dobles_fuente = 0
dobles_fisicas = 0
for tabla in tablas:
    df = tablas[tabla]
    faltantes = {"tipo_proceso_extraido", "tipo_proceso"} - set(df.columns)
    errores_correcciones += control(validaciones_correcciones, tabla + ": columnas de trazabilidad faltantes", 0, len(faltantes))
    if len(faltantes) > 0:
        continue
    fuente, inconsistentes = filas_fuente(df)
    fuentes[tabla] = fuente
    errores_correcciones += control(validaciones_correcciones, tabla + ": filas fuente longitudinales inconsistentes", 0, inconsistentes)
    dobles_fuente = dobles_fuente + int((contar_coincidencias_por_fila(fuente, tabla, "tipo_proceso_extraido") > 1).sum())
    dobles_fisicas = dobles_fisicas + int((contar_coincidencias_por_fila(df, tabla, "tipo_proceso_extraido") > 1).sum())
    errores_correcciones += control(validaciones_correcciones, tabla + ": filas fuente corregidas", FILAS_FUENTE_ESPERADAS[tabla],
                                    int(distintas(fuente["tipo_proceso_extraido"], fuente["tipo_proceso"]).sum()))
    errores_correcciones += control(validaciones_correcciones, tabla + ": filas físicas corregidas", FILAS_FISICAS_ESPERADAS[tabla],
                                    int(distintas(df["tipo_proceso_extraido"], df["tipo_proceso"]).sum()))
    errores_correcciones += control(validaciones_correcciones, tabla + ": dominio extraído", DOMINIOS_ANTES[tabla], int(df["tipo_proceso_extraido"].nunique()))
    errores_correcciones += control(validaciones_correcciones, tabla + ": dominio corregido", DOMINIOS_DESPUES[tabla], int(df["tipo_proceso"].nunique()))
    for regla in reglas_para_tabla(tabla):
        mascara = mascara_regla(fuente, regla, "tipo_proceso_extraido")
        observadas = int(mascara.sum())
        if observadas == 0:
            reglas_sin_match = reglas_sin_match + 1
        if observadas > regla["apariciones_fuente"]:
            reglas_con_exceso = reglas_con_exceso + 1
        if observadas == regla["apariciones_fuente"]:
            reglas_aplicadas = reglas_aplicadas + 1
        reglas_contradictorias = reglas_contradictorias + int((~(fuente.loc[mascara, "tipo_proceso"] == regla["literal_fuente"])).sum())

errores_correcciones += control(validaciones_correcciones, "reglas aplicadas", 87, reglas_aplicadas)
errores_correcciones += control(validaciones_correcciones, "reglas sin match", 0, reglas_sin_match)
errores_correcciones += control(validaciones_correcciones, "reglas con exceso", 0, reglas_con_exceso)
errores_correcciones += control(validaciones_correcciones, "reglas contradictorias", 0, reglas_contradictorias)
errores_correcciones += control(validaciones_correcciones, "filas fuente con más de una regla", 0, dobles_fuente)
errores_correcciones += control(validaciones_correcciones, "filas físicas con más de una regla", 0, dobles_fisicas)
total_fuente = 0
for d in fuentes.values():
    total_fuente = total_fuente + int(distintas(d["tipo_proceso_extraido"], d["tipo_proceso"]).sum())
total_fisicas = 0
dominio_combinado = set()
for d in tablas.values():
    total_fisicas = total_fisicas + int(distintas(d["tipo_proceso_extraido"], d["tipo_proceso"]).sum())
    dominio_combinado = dominio_combinado | set(d["tipo_proceso"].dropna())
errores_correcciones += control(validaciones_correcciones, "filas fuente corregidas", 635, total_fuente)
errores_correcciones += control(validaciones_correcciones, "filas físicas corregidas", 4280, total_fisicas)
errores_correcciones += control(validaciones_correcciones, "dominio combinado corregido", 147, len(dominio_combinado))

# Las variaciones editoriales (misma cosa escrita distinto) NO se corrigen.
editoriales = pd.read_csv(PROCESSED / "auditoria" / "propuesta_variaciones_editoriales_tipo_proceso.csv")
editoriales_aplicadas = 0
for i, fila in editoriales.iterrows():
    df = tablas[fila["tabla"]]
    mascara = (df["cuadro_origen"].astype(str) == str(fila["cuadro_origen"])) & (df["tipo_proceso_extraido"] == fila["literal_fuente"])
    editoriales_aplicadas = editoriales_aplicadas + int(distintas(df.loc[mascara, "tipo_proceso_extraido"], df.loc[mascara, "tipo_proceso"]).sum())
errores_correcciones += control(validaciones_correcciones, "variaciones editoriales aplicadas", 0, editoriales_aplicadas)

etapa, contexto, etapa_ok, contexto_ok = grupos_repetidos(causas)
errores_correcciones += control(validaciones_correcciones, "grupos por etapa diferenciados", 57, etapa_ok)
errores_correcciones += control(validaciones_correcciones, "grupos contexto penal diferenciados", 38, contexto_ok)
errores_correcciones += control(validaciones_correcciones, "cuadros inventariados", 95, len(cuadros))
errores_correcciones += control(validaciones_correcciones, "cuadros inesperados", 0, len(cuadros - CUADROS_CONOCIDOS))
estrato, conteos = estrato_semantico(causas)
duplicados = conteos[conteos > 1]
errores_correcciones += control(validaciones_correcciones, "filas estrato", 1997, len(estrato))
errores_correcciones += control(validaciones_correcciones, "claves semánticas únicas", 1997, len(conteos))
errores_correcciones += control(validaciones_correcciones, "grupos semánticos duplicados", 0, len(duplicados))
errores_correcciones += control(validaciones_correcciones, "filas semánticas duplicadas", 0, int(duplicados.sum()))
errores_correcciones += control(validaciones_correcciones, "máxima multiplicidad", 1, int(conteos.max()))
pd.DataFrame(validaciones_correcciones).to_csv(INTERIM / "validacion_correcciones_tipo_proceso.csv", index=False, encoding="utf-8")
print(len(validaciones_correcciones), "controles,", errores_correcciones, "fallos")

## Juzgados declarados dos veces
El 5.1.1.1 declara 153 juzgados civiles en capitales y el 9.1.1 declara 163. No se corrige: se registra.

In [ ]:
vistos = causas[causas["es_total_nacional"] & causas["num_juzgados_pagina"].notna()]
for cuadro, g in vistos.groupby("cuadro_origen"):
    if str(cuadro).startswith("5."):
        referencia = "9.1.1"
    else:
        referencia = "9.1.5"
    materias_cuadro = sorted(set(g["materia_norm"].dropna()))
    fila9 = mov[(mov["cuadro_origen"] == referencia) & mov["materia_norm"].isin(materias_cuadro)]
    if fila9.empty or fila9["num_juzgados"].isna().all():
        continue
    registrar("num_juzgados de " + cuadro + " = " + referencia, cuadro, g.iloc[0]["pagina_pdf"], " + ".join(materias_cuadro),
              float(g.iloc[0]["num_juzgados_pagina"]), float(fila9["num_juzgados"].sum()), "cruce entre capítulos")

## Controles geográficos de los capítulos 5 y 6
Cobertura de departamento, dominio, coherencia entidad/ámbito y cierre de cada bloque
territorial con su fila "TOTAL <entidad>".

In [ ]:
validaciones_geografia = []
inconsistencias_geografia = []


def registrar_inconsistencia_geografia(tipo, tabla, fila, detalle):
    inconsistencias_geografia.append({"validacion": tipo, "tabla": tabla, "cuadro_origen": fila.get("cuadro_origen"),
                                      "pagina_pdf": fila.get("pagina_pdf"), "entidad": fila.get("entidad"),
                                      "ambito": fila.get("ambito"), "departamento_derivado": fila.get("departamento_derivado"),
                                      "detalle": detalle})


departamentos = set(DEPARTAMENTOS)
dominios = {"capital": CAPITALES_NORM, "provincia": PROVINCIAS_NORM}
entidades_territoriales = CAPITALES_NORM | PROVINCIAS_NORM
for tabla in TABLAS_PROCESOS:
    df = pd.read_csv(INTERIM / TABLAS_PROCESOS[tabla], low_memory=False)
    nacional = df["es_total_nacional"].fillna(False).astype(bool)
    territorial = ~nacional
    con_departamento = territorial & df["departamento_derivado"].notna()
    sin_departamento = territorial & df["departamento_derivado"].isna()
    fuera_dominio = df["departamento_derivado"].notna() & ~df["departamento_derivado"].isin(departamentos)
    for i, fila in df[fuera_dominio].iterrows():
        registrar_inconsistencia_geografia("dominio_departamento", tabla, fila, "departamento_derivado no pertenece a los nueve departamentos")
    for i, fila in df[sin_departamento].iterrows():
        if not es_faltante(fila.get("entidad")) and fila.get("ambito") in dominios:
            registrar_inconsistencia_geografia("cobertura_departamento", tabla, fila, "fila territorial con información suficiente y sin departamento")
        else:
            registrar_inconsistencia_geografia("informacion_territorial", tabla, fila, "fila no nacional sin entidad o ámbito territorial válido")
    for i, fila in df[territorial].iterrows():
        ambito = fila.get("ambito")
        entidad = normalizar_geografia(fila.get("entidad"))
        if ambito not in dominios or entidad not in dominios[ambito]:
            registrar_inconsistencia_geografia("coherencia_ambito", tabla, fila, "la entidad no pertenece al dominio del ámbito derivado")
    # Los pocos alias del anuario se aceptan solo en el cuadro y la página donde se verificaron.
    totales = df[(df["tipo_fila_derivado"] == "total") & territorial].copy()
    claves_total = ["cuadro_origen", "pagina_pdf", "entidad", "tipo_proceso"]
    for i, fila in totales.drop_duplicates(claves_total).iterrows():
        rotulo = normalizar_geografia(fila.get("tipo_proceso"))
        if rotulo is None or not rotulo.startswith("TOTAL "):
            continue
        if rotulo[len("TOTAL "):] not in entidades_territoriales:
            continue
        if not total_corresponde_a_entidad(fila.get("cuadro_origen"), fila.get("pagina_pdf"), fila.get("ambito"),
                                           fila.get("entidad"), fila.get("tipo_proceso")):
            registrar_inconsistencia_geografia("total_entidad", tabla, fila,
                                               "el rótulo " + repr(fila.get("tipo_proceso")) + " no cierra el bloque de " + repr(fila.get("entidad")))
    validaciones_geografia.append({"tabla": tabla, "filas_totales": len(df), "filas_nacionales": int(nacional.sum()),
                                   "filas_territoriales": int(territorial.sum()),
                                   "filas_territoriales_con_departamento": int(con_departamento.sum()),
                                   "filas_territoriales_sin_departamento": int(sin_departamento.sum()),
                                   "departamentos_fuera_dominio": int(fuera_dominio.sum())})

resumen_geografia = pd.DataFrame(validaciones_geografia)
problemas_geografia = pd.DataFrame(inconsistencias_geografia, columns=["validacion", "tabla", "cuadro_origen", "pagina_pdf",
                                                                        "entidad", "ambito", "departamento_derivado", "detalle"])
resumen_geografia.to_csv(INTERIM / "validacion_geografia.csv", index=False, encoding="utf-8")
problemas_geografia.to_csv(INTERIM / "inconsistencias_geografia.csv", index=False, encoding="utf-8")
errores_geografia = len(problemas_geografia)
resumen_geografia

## Discrepancias del anuario

In [ ]:
disc = pd.DataFrame(discrepancias)
disc.to_csv(INTERIM / "discrepancias.csv", index=False, encoding="utf-8")
print("Discrepancias registradas:", len(disc), "(no se corrige ninguna)")
numericas = disc[disc["diferencia"].notna()]
numericas.groupby(["cuadro_origen", "identidad"]).size()

In [ ]:
por_identidad = numericas["identidad"].str.replace(r" \(.*\)$", "", regex=True).str.replace(r"^TOTAL .* = suma", "TOTAL <entidad> = suma", regex=True).value_counts()
plt.figure(figsize=(10, 4))
plt.barh(por_identidad.index, por_identidad.values, color="tab:red")
plt.gca().invert_yaxis()
plt.xlabel("discrepancias")
plt.title("Dónde el anuario no cierra consigo mismo")
plt.tight_layout()
plt.show()

In [ ]:
disc[disc["diferencia"].isna()]

In [ ]:
fallos = {"geografía": errores_geografia, "juzgados": errores_juzgados, "contexto": errores_contexto,
          "correcciones": errores_correcciones}
print(fallos)
if errores_geografia + errores_juzgados + errores_contexto + errores_correcciones > 0:
    raise RuntimeError("Fallaron validaciones técnicas: " + str(fallos))